Notebook to demonstrate the non-dimensionalization of the robust-diagnostic planetary-geostrophic equations

In [ ]:
using Symbolics

@variables x y z t

Dx = Differential(x)
Dy = Differential(y)
Dz = Differential(z)

@variables u(..) v(..) w(..) p(..) b(..) B(..)
@variables f(..) ν(..) κ(..)
@variables H(..)
@variables τx(..) τy(..)
@variables bs(..)

û = u(x,y,z)
v̂ = v(x,y,z)
ŵ = w(x,y,z)
p̂ = p(x,y,z)
b̂ = b(x,y,z)
B̂ = B(x,y,z)

f̂ = f(y)
ν̂ = ν(z)
κ̂ = κ(z)

Ĥ = H(x,y)
τ̂x = τx(x,y)
τ̂y = τy(x,y)
b̂s = bs(x,y)

bs(x, y)

In [2]:
@variables L H0 U0 ρ0 f0 N0 κ0 ν0 τ0 γ g

# Scales
Xscale = L
Yscale = L
Zscale = H0
Tscale = L / U0

Uscale = U0
Vscale = U0
Wscale = H0 * U0 / L

Pscale = ρ0 * f0 * L * U0
bscale = N0^2 * H0
Bscale = bscale

fscale = f0
νscale = ν0
κscale = κ0
τscale = τ0
Hscale = H0

dim_x(Q, q) = Q / L * Dx(q)
dim_y(Q, q) = Q / L * Dy(q)
dim_z(Q, q) = Q / H0 * Dz(q)
dim_t(Q, q) = Q / Tscale * Dt(q)

# Second vertical derivative of q* when q* = Q q̂:
dim_zz(Q, q) = Q / H0^2 * Dz(Dz(q))

vertical_visc_term(q) =
    (νscale * Uscale / H0^2) * Dz(ν̂ * Dz(q))

vertical_diff_term(q) =
    (κscale * bscale / H0^2) * Dz(κ̂ * Dz(q))

# Dimensional fields written in terms of nondimensional fields
u_star = Uscale * û
v_star = Uscale * v̂
w_star = Wscale * ŵ

p_star = Pscale * p̂
b_star = bscale * b̂
B_star = Bscale * B̂

f_star = fscale * f̂
ν_star = νscale * ν̂
κ_star = κscale * κ̂

κ(z)*κ0

In [3]:
eq_xmom_dim =
    -f_star * v_star ~
    -(1 / ρ0) * dim_x(Pscale, p̂) +
    vertical_visc_term(û)

eq_ymom_dim =
    f_star * u_star ~
    -(1 / ρ0) * dim_y(Pscale, p̂) +
    vertical_visc_term(v̂)

eq_cont_dim =
    dim_x(Uscale, û) +
    dim_y(Vscale, v̂) +
    dim_z(Wscale, ŵ) ~ 0

eq_hydro_dim =
    dim_z(Pscale, p̂) ~ ρ0 * (b_star - g)

eq_buoy_dim =
    u_star * dim_x(bscale, b̂) +
    v_star * dim_y(bscale, b̂) +
    w_star * dim_z(bscale, b̂) ~
    vertical_diff_term(b̂) +
    γ * (B_star - b_star)

(H0*(N0^2)*U0*Differential(x, 1)(b(x, y, z, t))*u(x, y, z, t)) / L + (H0*(N0^2)*U0*Differential(y, 1)(b(x, y, z, t))*v(x, y, z, t)) / L + (H0*(N0^2)*U0*w(x, y, z, t)*Differential(z, 1)(b(x, y, z, t))) / L ~ ((N0^2)*Differential(z, 1)(κ(z)*Differential(z, 1)(b(x, y, z, t)))*κ0) / H0 + (-H0*(N0^2)*b(x, y, z, t) + H0*(N0^2)*B(x, y, z, t))*γ

In [4]:
mom_scale = f0 * U0

eq_xmom_nd_raw = simplify(eq_xmom_dim.lhs / mom_scale ~ eq_xmom_dim.rhs / mom_scale)
eq_ymom_nd_raw = simplify(eq_ymom_dim.lhs / mom_scale ~ eq_ymom_dim.rhs / mom_scale)

eq_xmom_nd_raw, eq_ymom_nd_raw



(-f(x, y)*v(x, y, z, t) ~ (Differential(z, 1)(Differential(z, 1)(u(x, y, z, t))*ν(z))*ν0 - (H0^2)*f0*Differential(x, 1)(p(x, y, z, t))) / ((H0^2)*f0), u(x, y, z, t)*f(x, y) ~ (Differential(z, 1)(Differential(z, 1)(v(x, y, z, t))*ν(z))*ν0 - (H0^2)*f0*Differential(y, 1)(p(x, y, z, t))) / ((H0^2)*f0))

In [5]:
@variables ϵ²

subs_mom = Dict(
    ν0 / (f0 * H0^2) => ϵ²
)

eq_xmom_nd = simplify(substitute(eq_xmom_nd_raw, subs_mom))
eq_ymom_nd = simplify(substitute(eq_ymom_nd_raw, subs_mom))

eq_xmom_nd, eq_ymom_nd

(-f(x, y)*v(x, y, z, t) ~ (Differential(z, 1)(Differential(z, 1)(u(x, y, z, t))*ν(z))*ν0 - (H0^2)*f0*Differential(x, 1)(p(x, y, z, t))) / ((H0^2)*f0), u(x, y, z, t)*f(x, y) ~ (Differential(z, 1)(Differential(z, 1)(v(x, y, z, t))*ν(z))*ν0 - (H0^2)*f0*Differential(y, 1)(p(x, y, z, t))) / ((H0^2)*f0))

In [6]:
cont_scale = U0 / L

eq_cont_nd = simplify(eq_cont_dim.lhs / cont_scale ~ 0)

eq_cont_nd

Differential(z, 1)(w(x, y, z, t)) + Differential(x, 1)(u(x, y, z, t)) + Differential(y, 1)(v(x, y, z, t)) ~ 0